In [1]:
from pathlib import Path
import os

from IPython.display import display
from pathlib import Path
from rasterio import features
import os
import pandas as pd
from satellites.shared.constants import SEED_NUMBER
from satellites.shared.notebook import configure_notebook
configure_notebook(seed=42, tensorflow=True)
import satellites.shared.io as io_l

from satellites.ml_classification import (
    GeospatialClassificationPipeline,
    ClassificationPipelineConfig,
)

In [2]:
WORK_PATH = Path("C:/work_dir/neuro_2024-2025a")

PARCELS_PATH = WORK_PATH / "1_eo_parcel_stats_fill" / "satellite_parcel_time_stats.geoparquet"
EDA_REPORT_PATH = WORK_PATH / "2_eo_results_eda_fill" / "eda_annotated_data.parquet"
PROPOSED_FEATURES_PATH = WORK_PATH / "2_eo_results_eda_fill" / "tables" / "feature_selection" / "feature_selection_proposal.csv"
OUTPUT_PATH = WORK_PATH / "3_eo_ml_classification_fill"

class_column = "crop_category"
parcel_id_column = "parcel_id"


In [3]:
gdf_parcels = io_l.read_data(str(PARCELS_PATH))
eda_parcels = io_l.read_data(str(EDA_REPORT_PATH))
eda_parcels = eda_parcels[[parcel_id_column, "is_outlier_97_5pct"]]
gdf_parcels = gdf_parcels.merge(eda_parcels, on=parcel_id_column, how="left")
gdf_parcels.head(3)

Reading file: C:/work_dir/neuro_2024-2025a/1_eo_parcel_stats_fill/satellite_parcel_time_stats.geoparquet
Reading file: C:/work_dir/neuro_2024-2025a/2_eo_results_eda_fill/eda_annotated_data.parquet


,parcel_id,crop_category,cluster,geometry,batch_number,intersected_pixel_count,meets_minimum_pixel_count,expected_pixel_count,observed_count,temporal_filled_pixel_count,spatial_filled_pixel_count,temporal_filled_ratio,spatial_filled_ratio,data_reliability_score,pixel_area,geom_area,geom_interior_area_ratio,geom_compactness,geom_perimeter_area_ratio,geom_shape_index,geom_elongation,geom_shape_complexity_score,B02_coefficient_of_variation__20241001,B02_coefficient_of_variation__20241101,B02_coefficient_of_variation__20241201,B02_coefficient_of_variation__20250101,B02_coefficient_of_variation__20250201,B02_coefficient_of_variation__20250301,B02_coefficient_of_variation__20250401,B02_coefficient_of_variation__20250501,B02_coefficient_of_variation__20250601,B02_coefficient_of_variation__20250701,B02_coefficient_of_variation__20250801,B02_coefficient_of_variation__20250901,B02_max__20241001,B02_max__20241101,B02_max__20241201,B02_max__20250101,B02_max__20250201,B02_max__20250301,B02_max__20250401,B02_max__20250501,B02_max__20250601,B02_max__20250701,B02_max__20250801,B02_max__20250901,B02_mean__20241001,B02_mean__20241101,B02_mean__20241201,B02_mean__20250101,B02_mean__20250201,B02_mean__20250301,B02_mean__20250401,B02_mean__20250501,B02_mean__20250601,B02_mean__20250701,B02_mean__20250801,B02_mean__20250901,B02_median__20241001,B02_median__20241101,B02_median__20241201,B02_median__20250101,B02_median__20250201,B02_median__20250301,B02_median__20250401,B02_median__20250501,B02_median__20250601,B02_median__20250701,B02_median__20250801,B02_median__20250901,B02_min__20241001,B02_min__20241101,B02_min__20241201,B02_min__20250101,B02_min__20250201,B02_min__20250301,B02_min__20250401,B02_min__20250501,B02_min__20250601,B02_min__20250701,B02_min__20250801,B02_min__20250901,B02_range__20241001,B02_range__20241101,B02_range__20241201,B02_range__20250101,B02_range__20250201,B02_range__20250301,B02_range__20250401,B02_range__20250501,B02_range__20250601,B02_range__20250701,B02_range__20250801,B02_range__20250901,B02_sd__20241001,B02_sd__20241101,B02_sd__20241201,B02_sd__20250101,B02_sd__20250201,B02_sd__20250301,B02_sd__20250401,B02_sd__20250501,B02_sd__20250601,B02_sd__20250701,B02_sd__20250801,B02_sd__20250901,B02_variance__20241001,B02_variance__20241101,B02_variance__20241201,B02_variance__20250101,B02_variance__20250201,B02_variance__20250301,B02_variance__20250401,B02_variance__20250501,B02_variance__20250601,B02_variance__20250701,B02_variance__20250801,B02_variance__20250901,B03_coefficient_of_variation__20241001,B03_coefficient_of_variation__20241101,B03_coefficient_of_variation__20241201,B03_coefficient_of_variation__20250101,B03_coefficient_of_variation__20250201,B03_coefficient_of_variation__20250301,B03_coefficient_of_variation__20250401,B03_coefficient_of_variation__20250501,B03_coefficient_of_variation__20250601,B03_coefficient_of_variation__20250701,B03_coefficient_of_variation__20250801,B03_coefficient_of_variation__20250901,B03_max__20241001,B03_max__20241101,B03_max__20241201,B03_max__20250101,B03_max__20250201,B03_max__20250301,B03_max__20250401,B03_max__20250501,B03_max__20250601,B03_max__20250701,B03_max__20250801,B03_max__20250901,B03_mean__20241001,B03_mean__20241101,B03_mean__20241201,B03_mean__20250101,B03_mean__20250201,B03_mean__20250301,B03_mean__20250401,B03_mean__20250501,B03_mean__20250601,B03_mean__20250701,B03_mean__20250801,B03_mean__20250901,B03_median__20241001,B03_median__20241101,B03_median__20241201,B03_median__20250101,B03_median__20250201,B03_median__20250301,B03_median__20250401,B03_median__20250501,B03_median__20250601,B03_median__20250701,B03_median__20250801,B03_median__20250901,B03_min__20241001,B03_min__20241101,B03_min__20241201,B03_min__20250101,B03_min__20250201,B03_min__20250301,B03_min__20250401,B03_min__20250501,B03_min__20250601,B03_min__20250701,B03_min__20250801,B03_min__20250901,B03_range__20241001,B03_range__20241101,B03_range__20241201,B03_range__20250101,B03_range__20250201,B03_rang

In [4]:
gdf_parcels = gdf_parcels[gdf_parcels[class_column] != "jjjj"]
label_counts = gdf_parcels[class_column].value_counts()
print(f"Number of unique labels: {len(label_counts)}")
print(gdf_parcels[class_column].value_counts())
print(50*"-")
rare_labels = label_counts[label_counts < 5].index
gdf_parcels.loc[gdf_parcels[class_column].isin(rare_labels), class_column] = "9999"
print(f"Number of unique labels after grouping rare labels: {len(gdf_parcels[class_column].value_counts())}")
print(gdf_parcels[class_column].value_counts())

cond = gdf_parcels[class_column] == "9999"
print(50*"-")
print(f"Excluded parcels with label '9999': {cond.sum()} out of {len(gdf_parcels)}")
gdf_parcels = gdf_parcels[~cond].copy()

Number of unique labels: 25
crop_category
ΚΤΗΝΟΤΡΟΦΙΚΑ ΦΥΤΑ                                       305
ΒΑΜΒΑΚΙ                                                 224
ΣΚΛΗΡΟΣ ΣΙΤΟΣ                                           149
ΔΑΣΙΚΑ ΔΕΝΤΡΑ                                            95
ΕΝΕΡΓΕΙΑΚΕΣ ΚΑΛΛΙΕΡΓΕΙΕΣ                                 82
ΒΟΣΚΟΤΟΠΟΣ                                               61
ΛΟΙΠΑ ΣΙΤΗΡΑ                                             50
ΑΡΩΜΑΤΙΚΑ ΦΥΤΑ                                           49
ΑΓΡΑΝΑΠΑΥΣΗ                                              41
ΕΛΑΙΩΝΕΣ                                                 26
ΓΗ ΠΟΥ ΔΕΝ ΕΝΤΑΣΣΕΤΑΙ ΣΕ ΚΑΛΛΙΕΡΓΗΤΙΚΗ ΔΡΑΣΤΗΡΙΟΤΗΤΑ     13
ΑΜΠΕΛΩΝΕΣ                                                11
ΑΚΡΟΔΡΥΑ (ΚΑΡΠΟΙ ΜΕ ΚΕΛΥΦΟΣ)                              9
ΑΡΑΒΟΣΙΤΟΣ                                                8
ΟΣΠΡΙΑ ΒΡΩΣΙΜΑ                                            7
ΠΟΡΤΟΚΑΛΙΑ ΠΡΟΣ ΧΥΜΟΠΟΙΗΣΗ                                

In [5]:
df_features = io_l.read_data(str(PROPOSED_FEATURES_PATH))
display(df_features.head())

proposed_features = df_features[df_features["proposed_action"]=="use"]["feature"].tolist()
print(f"Proposed features: {len(proposed_features)}")
proposed_features

Reading file: C:/work_dir/neuro_2024-2025a/2_eo_results_eda_fill/tables/feature_selection/feature_selection_proposal.csv


,selection_order,feature,feature_type,proposed_action,primary_reason,association_method,effect_size,association_strength,qvalue,missing_percent,redundant_with,redundancy_measure,passes_hard_exclusions,passes_data_quality,passes_target_evidence,passes_redundancy,passes_shortlist_limit
0,1,GAIOIKANOT,categorical,use,priority target ...,bias_corrected_c...,0.4060000000,moderate,0.0000000000,0.0000000000,NaN,NaN,True,True,True,True,True
1,2,elevation_mean_m,numeric,use,priority target ...,kruskal_epsilon_...,0.3360000000,large,0.0000000000,0.0000000000,NaN,NaN,True,True,True,True,True
2,3,NDWI_median__202...,numeric,use,priority target ...,kruskal_epsilon_...,0.3114000000,large,0.0000000000,0.0000000000,NaN,NaN,True,True,True,True,True
3,4,NDVI_median__202...,numeric,use,priority target ...,kruskal_epsilon_...,0.2866000000,large,0.0000000000,0.0858000000,NaN,NaN,True,True,True,True,True
4,5,NDVI_median__202...,numeric,use,priority target ...,kruskal_epsilon_...,0.2835000000,large,0.0000000000,0.0000000000,NaN,NaN,True,True,True,True,True


Proposed features: 28


['GAIOIKANOT',
 'elevation_mean_m',
 'NDWI_median__20241001',
 'NDVI_median__20241201',
 'NDVI_median__20241101',
 'NDMI_median__20241001',
 'EVI_median__20241001',
 'NDMI_median__20241101',
 'EVI_median__20241201',
 'NDVI_median__20250901',
 'NDVI_median__20250701',
 'NDRE_median__20250101',
 'NDRE_median__20250301',
 'NDVI_median__20250401',
 'NDVI_median__20250501',
 'NDVI_median__20250601',
 'NDWI_median__20250101',
 'NDMI_median__20250101',
 'NDVI_median__20250201',
 'NDMI_median__20250201',
 'EVI_median__20250101',
 'NDMI_median__20250301',
 'EVI_median__20250201',
 'NDMI_median__20241201',
 'NDMI_median__20250901',
 'NDMI_median__20250701',
 'data_reliability_score',
 'NDMI_median__20250601']

In [6]:
gdf_parcels = gdf_parcels.to_crs(2100)
gdf_parcels["x"] = gdf_parcels.centroid.x
gdf_parcels["y"] = gdf_parcels.centroid.y

base_cols = [parcel_id_column, class_column, "geometry", "data_reliability_score", "geom_shape_complexity_score"]
extra_features = ["elevation_mean_m", "GAIOIKANOT", "x", "y"]
temporal_features = ("_median", "_sd", "_range",)
selected_indices = ("NDMI", "NDRE", "SAVI", "NDVI", "EVI", "NDWI",)
original_features = [x for x in gdf_parcels.columns if any(keyword in x for keyword in temporal_features) and any(idx in x for idx in selected_indices)]

gdf_parcels = gdf_parcels[base_cols + original_features + extra_features].copy()
features = original_features + extra_features

gdf_parcels = gdf_parcels[gdf_parcels[class_column] != 9999].copy()
print(f"Final features for modeling ({len(features)} features): {features}")
display(gdf_parcels.head(3))

Final features for modeling (220 features): ['EVI_median__20241001', 'EVI_median__20241101', 'EVI_median__20241201', 'EVI_median__20250101', 'EVI_median__20250201', 'EVI_median__20250301', 'EVI_median__20250401', 'EVI_median__20250501', 'EVI_median__20250601', 'EVI_median__20250701', 'EVI_median__20250801', 'EVI_median__20250901', 'EVI_range__20241001', 'EVI_range__20241101', 'EVI_range__20241201', 'EVI_range__20250101', 'EVI_range__20250201', 'EVI_range__20250301', 'EVI_range__20250401', 'EVI_range__20250501', 'EVI_range__20250601', 'EVI_range__20250701', 'EVI_range__20250801', 'EVI_range__20250901', 'EVI_sd__20241001', 'EVI_sd__20241101', 'EVI_sd__20241201', 'EVI_sd__20250101', 'EVI_sd__20250201', 'EVI_sd__20250301', 'EVI_sd__20250401', 'EVI_sd__20250501', 'EVI_sd__20250601', 'EVI_sd__20250701', 'EVI_sd__20250801', 'EVI_sd__20250901', 'NDMI_median__20241001', 'NDMI_median__20241101', 'NDMI_median__20241201', 'NDMI_median__20250101', 'NDMI_median__20250201', 'NDMI_median__20250301', '

,parcel_id,crop_category,geometry,data_reliability_score,geom_shape_complexity_score,EVI_median__20241001,EVI_median__20241101,EVI_median__20241201,EVI_median__20250101,EVI_median__20250201,EVI_median__20250301,EVI_median__20250401,EVI_median__20250501,EVI_median__20250601,EVI_median__20250701,EVI_median__20250801,EVI_median__20250901,EVI_range__20241001,EVI_range__20241101,EVI_range__20241201,EVI_range__20250101,EVI_range__20250201,EVI_range__20250301,EVI_range__20250401,EVI_range__20250501,EVI_range__20250601,EVI_range__20250701,EVI_range__20250801,EVI_range__20250901,EVI_sd__20241001,EVI_sd__20241101,EVI_sd__20241201,EVI_sd__20250101,EVI_sd__20250201,EVI_sd__20250301,EVI_sd__20250401,EVI_sd__20250501,EVI_sd__20250601,EVI_sd__20250701,EVI_sd__20250801,EVI_sd__20250901,NDMI_median__20241001,NDMI_median__20241101,NDMI_median__20241201,NDMI_median__20250101,NDMI_median__20250201,NDMI_median__20250301,NDMI_median__20250401,NDMI_median__20250501,NDMI_median__20250601,NDMI_median__20250701,NDMI_median__20250801,NDMI_median__20250901,NDMI_range__20241001,NDMI_range__20241101,NDMI_range__20241201,NDMI_range__20250101,NDMI_range__20250201,NDMI_range__20250301,NDMI_range__20250401,NDMI_range__20250501,NDMI_range__20250601,NDMI_range__20250701,NDMI_range__20250801,NDMI_range__20250901,NDMI_sd__20241001,NDMI_sd__20241101,NDMI_sd__20241201,NDMI_sd__20250101,NDMI_sd__20250201,NDMI_sd__20250301,NDMI_sd__20250401,NDMI_sd__20250501,NDMI_sd__20250601,NDMI_sd__20250701,NDMI_sd__20250801,NDMI_sd__20250901,NDRE_median__20241001,NDRE_median__20241101,NDRE_median__20241201,NDRE_median__20250101,NDRE_median__20250201,NDRE_median__20250301,NDRE_median__20250401,NDRE_median__20250501,NDRE_median__20250601,NDRE_median__20250701,NDRE_median__20250801,NDRE_median__20250901,NDRE_range__20241001,NDRE_range__20241101,NDRE_range__20241201,NDRE_range__20250101,NDRE_range__20250201,NDRE_range__20250301,NDRE_range__20250401,NDRE_range__20250501,NDRE_range__20250601,NDRE_range__20250701,NDRE_range__20250801,NDRE_range__20250901,NDRE_sd__20241001,NDRE_sd__20241101,NDRE_sd__20241201,NDRE_sd__20250101,NDRE_sd__20250201,NDRE_sd__20250301,NDRE_sd__20250401,NDRE_sd__20250501,NDRE_sd__20250601,NDRE_sd__20250701,NDRE_sd__20250801,NDRE_sd__20250901,NDVI_median__20241001,NDVI_median__20241101,NDVI_median__20241201,NDVI_median__20250101,NDVI_median__20250201,NDVI_median__20250301,NDVI_median__20250401,NDVI_median__20250501,NDVI_median__20250601,NDVI_median__20250701,NDVI_median__20250801,NDVI_median__20250901,NDVI_range__20241001,NDVI_range__20241101,NDVI_range__20241201,NDVI_range__20250101,NDVI_range__20250201,NDVI_range__20250301,NDVI_range__20250401,NDVI_range__20250501,NDVI_range__20250601,NDVI_range__20250701,NDVI_range__20250801,NDVI_range__20250901,NDVI_sd__20241001,NDVI_sd__20241101,NDVI_sd__20241201,NDVI_sd__20250101,NDVI_sd__20250201,NDVI_sd__20250301,NDVI_sd__20250401,NDVI_sd__20250501,NDVI_sd__20250601,NDVI_sd__20250701,NDVI_sd__20250801,NDVI_sd__20250901,NDWI_median__20241001,NDWI_median__20241101,NDWI_median__20241201,NDWI_median__20250101,NDWI_median__20250201,NDWI_median__20250301,NDWI_median__20250401,NDWI_median__20250501,NDWI_median__20250601,NDWI_median__20250701,NDWI_median__20250801,NDWI_median__20250901,NDWI_range__20241001,NDWI_range__20241101,NDWI_range__20241201,NDWI_range__20250101,NDWI_range__20250201,NDWI_range__20250301,NDWI_range__20250401,NDWI_range__20250501,NDWI_range__20250601,NDWI_range__20250701,NDWI_range__20250801,NDWI_range__20250901,NDWI_sd__20241001,NDWI_sd__20241101,NDWI_sd__20241201,NDWI_sd__20250101,NDWI_sd__20250201,NDWI_sd__20250301,NDWI_sd__20250401,NDWI_sd__20250501,NDWI_sd__20250601,NDWI_sd__20250701,NDWI_sd__20250801,NDWI_sd__20250901,SAVI_median__20241001,SAVI_median__20241101,SAVI_median__20241201,SAVI_median__20250101,SAVI_median__20250201,SAVI_median__20250301,SAVI_median__20250401,SAVI_median__20250501,SAVI_median__20250601,SAVI_median__20250701,SAVI_median__20250801,SAVI_median__20250901,SAVI_range__20241001,SAVI_r

In [13]:
config = ClassificationPipelineConfig(
    project_dir=OUTPUT_PATH,
    target_column=class_column,
    feature_columns=features,
    id_column=parcel_id_column,

    apply_iqr=False,
    spatial_interpolation_method="nearest",  # None, nearest, idw, or kriging

    cv_folds=5,
    test_size=0.2,
    spatial_split=True,
    spatial_split_grid_size=30,
    spatial_split_method="by_row",
    random_state=SEED_NUMBER,

    scoring_primary="f1_macro",
    cv_ranking_method="score_minus_std",
    selection_type="soft_voting",
    top_voting_models=8,
    max_search_candidates=40,
    prediction_confidence_threshold=0.60,

    interpretability_top_models=7,
    interpretability_include_shap=False,

    optimize_class_probabilities=True,
    probability_multiplier_grid=(0.8, 1.0, 1.2, 1.5, 2.0),
    probability_optimization_iterations=2,
    probability_optimization_max_accuracy_drop=0.02,

    rank_confidence_enabled=True,
    force_retrain_models=False,
    class_reliability_enabled=False,

    label_balancing_method="random_oversample",
    lstm_temporal_statistics=temporal_features,
    selected_models=(
        "bagging",
        "bayesian",
        "decision_tree",
        "extra_trees",
        "random_forest",
        "gradient_boosting",
        "lightgbm",
        "xgboost",
        "support_vector_machine",
        "knn",
        "neural_network",
        # "keras_lstm",
   ),
    max_map_geometries=20000,
    open_html_report=True,
)

layer_features = max(16, min(len(features), 64))
config.tune_params["tensorflow_neural_network"]["model__hidden_layer_sizes"] = [(layer_features,), (layer_features, layer_features), (layer_features, layer_features // 2)]
config.tune_params["neural_network"]["model__hidden_layer_sizes"] = [(layer_features,), (layer_features, layer_features), (layer_features, layer_features // 2)]
config.tune_params["keras_lstm"]["model__lstm_units_1"] = [16, 32, 64]
pipeline = GeospatialClassificationPipeline(config)


In [8]:
summary_check = pipeline.run_check(df=gdf_parcels)
summary_check


Function <run_check> started on 2026-09-28 15:11:31

Checking input data...

Resetting project output folder: C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill
Deleting folder 'C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\01_check'...
Successfully deleted folder 'C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\01_check' on attempt 1
Deleting folder 'C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\02_prepare'...
Successfully deleted folder 'C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\02_prepare' on attempt 1
Deleting folder 'C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\03_train'...
Successfully deleted folder 'C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\03_train' on attempt 1
Deleting folder 'C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\04_evaluate'...
Successfully deleted folder 'C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\04_evaluate' on attempt 1
Deleting folder 'C:\work_dir\neuro_202

{'rows_total': 1150,
 'rows_labeled': 1150,
 'rows_unknown_target': 0,
 'n_features_requested': 220,
 'n_features_active': 220,
 'numeric_features': ['EVI_median__20241001',
  'EVI_median__20241101',
  'EVI_median__20241201',
  'EVI_median__20250101',
  'EVI_median__20250201',
  'EVI_median__20250301',
  'EVI_median__20250401',
  'EVI_median__20250501',
  'EVI_median__20250601',
  'EVI_median__20250701',
  'EVI_median__20250801',
  'EVI_median__20250901',
  'EVI_range__20241001',
  'EVI_range__20241101',
  'EVI_range__20241201',
  'EVI_range__20250101',
  'EVI_range__20250201',
  'EVI_range__20250301',
  'EVI_range__20250401',
  'EVI_range__20250501',
  'EVI_range__20250601',
  'EVI_range__20250701',
  'EVI_range__20250801',
  'EVI_range__20250901',
  'EVI_sd__20241001',
  'EVI_sd__20241101',
  'EVI_sd__20241201',
  'EVI_sd__20250101',
  'EVI_sd__20250201',
  'EVI_sd__20250301',
  'EVI_sd__20250401',
  'EVI_sd__20250501',
  'EVI_sd__20250601',
  'EVI_sd__20250701',
  'EVI_sd__20250801'

In [9]:
summary_prepare = pipeline.run_prepare()
summary_prepare


Function <run_prepare> started on 2026-09-28 15:11:33

Preparing data...


c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/02_prepare/label_mapping.csv

Prepared train/test split: 920 train rows, 230 test rows.

Execution time of function <run_prepare> ended on 2026-09-28 15:11:35 - duration: 00:00:02


{'active_features': ['EVI_median__20241001',
  'EVI_median__20241101',
  'EVI_median__20241201',
  'EVI_median__20250101',
  'EVI_median__20250201',
  'EVI_median__20250301',
  'EVI_median__20250401',
  'EVI_median__20250501',
  'EVI_median__20250601',
  'EVI_median__20250701',
  'EVI_median__20250801',
  'EVI_median__20250901',
  'EVI_range__20241001',
  'EVI_range__20241101',
  'EVI_range__20241201',
  'EVI_range__20250101',
  'EVI_range__20250201',
  'EVI_range__20250301',
  'EVI_range__20250401',
  'EVI_range__20250501',
  'EVI_range__20250601',
  'EVI_range__20250701',
  'EVI_range__20250801',
  'EVI_range__20250901',
  'EVI_sd__20241001',
  'EVI_sd__20241101',
  'EVI_sd__20241201',
  'EVI_sd__20250101',
  'EVI_sd__20250201',
  'EVI_sd__20250301',
  'EVI_sd__20250401',
  'EVI_sd__20250501',
  'EVI_sd__20250601',
  'EVI_sd__20250701',
  'EVI_sd__20250801',
  'EVI_sd__20250901',
  'NDMI_median__20241001',
  'NDMI_median__20241101',
  'NDMI_median__20241201',
  'NDMI_median__20250101

In [10]:
summary_train = pipeline.run_train()
summary_train


Function <run_train> started on 2026-09-28 15:11:35

Training models...
Reading file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/02_prepare/label_mapping.csv
____________________________________________________________________________________________________

Training model support_vector_machine


c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/03_train/models/support_vector_machine/cv_results.csv

Saved artifacts for support_vector_machine: model=C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\03_train\models\support_vector_machine\best_model.joblib, cv_results=C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\03_train\models\support_vector_machine\cv_results.csv, plot=C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\03_train\models\support_vector_machine\search_results.png

Model support_vector_machine finished in duration: 00:00:10
____________________________________________________________________________________________________

Training model random_forest
Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/03_train/models/random_forest/cv_results.csv

Saved artifacts for random_forest: model=C:\work_dir\neuro_2024-2025a\3_eo_ml_classification_fill\03_train\models\random_forest\best_model.joblib, cv_results

{'models_trained': ['xgboost',
  'random_forest',
  'bagging',
  'extra_trees',
  'lightgbm',
  'support_vector_machine',
  'gradient_boosting',
  'neural_network',
  'knn',
  'bayesian',
  'decision_tree'],
 'models_failed': [],
 'best_cv_model': 'xgboost',
 'cv_ranking_method': 'score_minus_std'}

In [14]:
summary_evaluate = pipeline.run_evaluate()
summary_evaluate


Function <run_evaluate> started on 2026-09-28 16:01:32

Evaluating models...
Reading file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/02_prepare/label_mapping.csv
Reading file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/03_train/training_summary.csv


c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



Removing existing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/reports/support_vector_machine_classification_report.csv
Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/reports/support_vector_machine_classification_report.csv


c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



Removing existing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/reports/random_forest_classification_report.csv
Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/reports/random_forest_classification_report.csv


...

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



Removing existing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/reports/xgboost_classification_report.csv
Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/reports/xgboost_classification_report.csv


c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



Removing existing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/reports/lightgbm_classification_report.csv
Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/reports/lightgbm_classification_report.csv
Removing existing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/model_metrics_train.csv
Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/model_metrics_train.csv
Removing existing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/model_metrics_test.csv
Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/model_metrics_test.csv
Removing existing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/interpretability/xgboost_feature_importance.csv
Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/04_evaluate/interpretability/xgboost_feature_importance.csv
Removing existing fi

{'selection_type': 'soft_voting',
 'selected_models': ['xgboost',
  'random_forest',
  'bagging',
  'extra_trees',
  'lightgbm',
  'support_vector_machine',
  'gradient_boosting',
  'neural_network'],
 'selected_metric': 'cv_score_minus_std',
 'selected_score': 0.12475514546844316,
 'selection_source': 'cross_validation',
 'labels': ['KHPEYTIKA',
  'ΑΓΡΑΝΑΠΑΥΣΗ',
  'ΑΚΡΟΔΡΥΑ (ΚΑΡΠΟΙ ΜΕ ΚΕΛΥΦΟΣ)',
  'ΑΜΠΕΛΩΝΕΣ',
  'ΑΡΑΒΟΣΙΤΟΣ',
  'ΑΡΩΜΑΤΙΚΑ ΦΥΤΑ',
  'ΒΑΜΒΑΚΙ',
  'ΒΟΣΚΟΤΟΠΟΣ',
  'ΓΗ ΠΟΥ ΔΕΝ ΕΝΤΑΣΣΕΤΑΙ ΣΕ ΚΑΛΛΙΕΡΓΗΤΙΚΗ ΔΡΑΣΤΗΡΙΟΤΗΤΑ',
  'ΔΑΣΙΚΑ ΔΕΝΤΡΑ',
  'ΕΛΑΙΟΥΧΟΙ ΣΠΟΡΟΙ',
  'ΕΛΑΙΩΝΕΣ',
  'ΕΝΕΡΓΕΙΑΚΕΣ ΚΑΛΛΙΕΡΓΕΙΕΣ',
  'ΚΤΗΝΟΤΡΟΦΙΚΑ ΦΥΤΑ',
  'ΛΟΙΠΑ ΣΙΤΗΡΑ',
  'ΟΣΠΡΙΑ ΒΡΩΣΙΜΑ',
  'ΠΟΡΤΟΚΑΛΙΑ ΠΡΟΣ ΧΥΜΟΠΟΙΗΣΗ',
  'ΠΥΡΗΝΟΚΑΡΠΑ',
  'ΣΚΛΗΡΟΣ ΣΙΤΟΣ'],
 'class_probability_multipliers': {'KHPEYTIKA': 1.0,
  'ΑΓΡΑΝΑΠΑΥΣΗ': 1.5,
  'ΑΚΡΟΔΡΥΑ (ΚΑΡΠΟΙ ΜΕ ΚΕΛΥΦΟΣ)': 1.2,
  'ΑΜΠΕΛΩΝΕΣ': 1.5,
  'ΑΡΑΒΟΣΙΤΟΣ': 1.0,
  'ΑΡΩΜΑΤΙΚΑ ΦΥΤΑ': 2.0,
  'ΒΑΜΒΑΚΙ': 0.8,
  'ΒΟΣΚΟΤΟΠΟΣ': 0.8,
  'ΓΗ ΠΟΥ ΔΕΝ ΕΝΤΑΣΣΕΤΑΙ ΣΕ 

In [15]:
summary_predict = pipeline.run_predict()
summary_predict


Function <run_predict> started on 2026-09-28 16:01:48

Classify all data...
Reading file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/02_prepare/label_mapping.csv


c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning:

The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`



Removing existing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/05_predict/final_predictions_preview.csv
Writing file: C:/work_dir/neuro_2024-2025a/3_eo_ml_classification_fill/05_predict/final_predictions_preview.csv

Filled 0 unknown labels.

Execution time of function <run_predict> ended on 2026-09-28 16:03:33 - duration: 00:01:44


{'selection_type': 'soft_voting',
 'selected_models': ['xgboost',
  'random_forest',
  'bagging',
  'extra_trees',
  'lightgbm',
  'support_vector_machine',
  'gradient_boosting',
  'neural_network'],
 'rows_filled': 0,
 'rows_unknown_original': 0,
 'rows_needing_review': 57,
 'prediction_max_probability_column': 'prediction_max_probability',
 'review_decision_basis': 'rank_confidence',
 'rank_confidence_enabled': True,
 'rank_confidence_contract_source': 'selection_summary',
 'confidence_method': 'rank_consensus_only',
 'class_reliability_enabled': False,
 'class_reliability_applied_to_final_confidence': False,
 'class_reliability_method': 'oof_precision',
 'final_confidence_rule': 'rank_only',
 'rank_confidence_minimum_models': 3,
 'rank_confidence_level_counts': {'HIGH': 1070, 'LOW': 57, 'MEDIUM': 23},
 'output_path': 'C:\\work_dir\\neuro_2024-2025a\\3_eo_ml_classification_fill\\05_predict\\final_predictions.joblib',
 'inspection_scoring_enabled': True,
 'inspection_scoring_availabl